In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import re


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/african-folktales-slm-challenge/sample_submission.csv
/kaggle/input/competitions/african-folktales-slm-challenge/train_prompts.csv
/kaggle/input/competitions/african-folktales-slm-challenge/documents.csv
/kaggle/input/competitions/african-folktales-slm-challenge/dataset-metadata.json
/kaggle/input/competitions/african-folktales-slm-challenge/test_prompts.csv
/kaggle/input/competitions/african-folktales-slm-challenge/baseline_submission.csv


In [2]:
DATA_DIR = "/kaggle/input/competitions/african-folktales-slm-challenge"

documents = pd.read_csv(
    os.path.join(DATA_DIR, "documents.csv")
)

train = pd.read_csv(
    os.path.join(DATA_DIR, "train_prompts.csv")
)

test = pd.read_csv(
    os.path.join(DATA_DIR, "test_prompts.csv")
)

sample_submission = pd.read_csv(
    os.path.join(DATA_DIR, "sample_submission.csv")
)

baseline = pd.read_csv(
    os.path.join(DATA_DIR, "baseline_submission.csv")
)

In [3]:
def clean_text(text):
    text = str(text).lower()

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
document_candidates = documents[[
    "document_id",
    "title",
    "text",
    "theme",
    "culture_region"
]].copy()

document_candidates["source"] = "document"

train_candidates = train[[
    "document_id",
    "reference_story",
    "theme",
    "culture_region"
]].copy()

train_candidates = train_candidates.rename(
    columns={
        "reference_story": "text"
    }
)

train_candidates["title"] = "training_reference"
train_candidates["source"] = "train_reference"

train_candidates = train_candidates[[
    "document_id",
    "title",
    "text",
    "theme",
    "culture_region",
    "source"
]]

In [5]:
candidates = pd.concat(
    [document_candidates, train_candidates],
    ignore_index=True
)

print(candidates.shape)

(62, 6)


In [6]:
test["retrieval_text"] = (
    test["prompt"].fillna("") + " " +
    test["theme"].fillna("") + " " +
    test["culture_region"].fillna("")
)

test["retrieval_text"] = test["retrieval_text"].apply(clean_text)

In [7]:
candidates["retrieval_text"] = (
    candidates["title"].fillna("") + " " +
    candidates["text"].fillna("") + " " +
    candidates["theme"].fillna("") + " " +
    candidates["culture_region"].fillna("")
)

candidates["retrieval_text"] = (
    candidates["retrieval_text"]
    .apply(clean_text)
)

In [8]:
candidate_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True
)

candidate_vectors = candidate_vectorizer.fit_transform(
    candidates["retrieval_text"]
)

In [9]:
test_vectors = candidate_vectorizer.transform(
    test["retrieval_text"]
)

In [10]:
candidate_similarity = cosine_similarity(
    test_vectors,
    candidate_vectors
)

In [11]:
for i, row in test.iterrows():

    scores = candidate_similarity[i]

    best_indices = np.argsort(scores)[::-1][:5]

    print("=" * 80)
    print("PROMPT:", row["prompt"])

    for rank, idx in enumerate(best_indices, 1):

        candidate = candidates.iloc[idx]

        print(
            f"{rank}. "
            f"{candidate['source']} | "
            f"{candidate['title']} | "
            f"{scores[idx]:.4f}"
        )

PROMPT: Tell a moral market tale about returning a lost cowrie shell.
1. train_reference | training_reference | 0.4451
2. document | The girl who returned the lost cowrie | 0.2881
3. train_reference | training_reference | 0.1054
4. train_reference | training_reference | 0.0959
5. train_reference | training_reference | 0.0637
PROMPT: Create a hero story of an orphan who saves fishermen in a squall.
1. train_reference | training_reference | 0.2741
2. document | Sail of the lake orphan | 0.2589
3. train_reference | training_reference | 0.1109
4. train_reference | training_reference | 0.1024
5. train_reference | training_reference | 0.0861
PROMPT: Create an East African tale of hare claiming thunder for himself.
1. train_reference | training_reference | 0.2973
2. document | Hare and the drum of thunder | 0.2539
3. train_reference | training_reference | 0.1090
4. document | Sail of the lake orphan | 0.0972
5. train_reference | training_reference | 0.0865
PROMPT: Hyena jumps at the moon in w